# PersistBench — Week 2 formation pilot (Kaggle)

Runs `scripts/run_pilot.py` with `configs/pilot.json`: 2 models (Qwen/Qwen2.5-7B-Instruct, Qwen/Qwen3-8B) x 6 chains x 5 fault conditions x memory policies M0-M5, measuring Session-1 **false-memory formation rates**. Full procedure: `kaggle/RUNBOOK.md` in the repo.

**PREREQUISITE — read first:** the PersistBench repo is currently **local-only** (no GitHub remote). Either (A) publish it to GitHub first and set `PERSISTBENCH_REPO_URL` in the code-acquisition cell, or (B) upload the repo folder (`persistbench/`, `configs/`, `scripts/`) as a Kaggle dataset / direct upload so it lands at `/kaggle/working/persistbench`. The acquisition cell hard-fails with instructions if neither is present.

**Gating:** launch only when the GPU sequencing clears (this pilot queues after the latent paper's staged Kaggle runs).

## 1. GPU check — hard fail without CUDA

The pilot models are 7-8B at 4-bit; a CPU run would take days and a silent Accelerator=None commit must never burn a session. This cell raises if no GPU is visible.

In [ ]:
import torch

print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'NO GPU: set Accelerator = GPU T4 x2 before Save & Run All, then re-run.'
print('device:', torch.cuda.get_device_name(0))

## 2. Get the code

Option A: public GitHub repo (set the URL). Option B: the repo uploaded into the working directory. The cell clones into `/kaggle/working/persistbench` or uses the uploaded copy, then prints the commit it found (the repo must be the Week 2 pilot package: `persistbench/llm_agent.py` and `configs/pilot.json` present).

In [ ]:
import os, subprocess

REPO_URL = os.environ.get('PERSISTBENCH_REPO_URL', '')  # set if the repo has been published
DEST = '/kaggle/working/persistbench'

if REPO_URL:
    subprocess.run(['git', 'clone', REPO_URL, DEST], check=True)
elif os.path.exists(os.path.join(DEST, 'configs', 'pilot.json')):
    print('using uploaded repo at', DEST)
else:
    raise RuntimeError(
        'PersistBench code not found. Publish the local repo to GitHub and set '
        'PERSISTBENCH_REPO_URL, or upload the repo folder so that '
        '/kaggle/working/persistbench/configs/pilot.json exists. See kaggle/RUNBOOK.md.')

assert os.path.exists(os.path.join(DEST, 'persistbench', 'llm_agent.py')), 'pilot package missing from repo copy'
subprocess.run(['git', '-C', DEST, 'log', '--oneline', '-1'])
print('repo ready at', DEST)

## 3. Install dependencies — preserving Kaggle's CUDA torch

Kaggle's image ships the CUDA torch build the pilot needs; a naive install can replace it. The packages below do not pin torch, and the cell re-checks CUDA afterwards and hard-fails if the GPU build was clobbered. (`transformers` serves the models, `accelerate` handles device placement, `bitsandbytes` provides 4-bit loading.)

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'accelerate', 'bitsandbytes'], check=True)

import torch
assert torch.cuda.is_available(), 'CUDA torch was clobbered by the install — stop and see RUNBOOK.'
print('deps installed; cuda still available:', torch.cuda.get_device_name(0))

## 4. Run the pilot

60 origin runs (2 models x 6 chains x 5 faults), each shared by all 6 memory policies, plus one memory-write generation per agent-authored policy. Results land in `/kaggle/working/pilot-results` so they persist in the version Output even if the session is later killed at the 12 h cap.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, 'scripts/run_pilot.py',
     '--config', 'configs/pilot.json',
     '--out', '/kaggle/working/pilot-results'],
    cwd='/kaggle/working/persistbench')
assert result.returncode == 0, 'pilot run failed — see output above'

## 5. Outputs — retrieval list

Expected files in `/kaggle/working/pilot-results` (also the version Output tab): `pilot_summary.md` (formation table), `pilot_formation.csv`, `pilot_cells.csv`, `pilot_cells.jsonl`, `pilot_writes.jsonl` (raw summaries + stored writes + truth), `pilot_summary.json`, `handlabel_sample.csv` (50 writes for the classifier-validation gate), `handlabel_instructions.md`.

In [ ]:
import os

OUT = '/kaggle/working/pilot-results'
for name in sorted(os.listdir(OUT)):
    print(f'{name:32} {os.path.getsize(os.path.join(OUT, name)):>10} bytes')

print()
print(open(os.path.join(OUT, 'pilot_summary.md')).read())

## After retrieval (not in this notebook)

1. Hand-label the 50-write sample per `handlabel_instructions.md`; the automatic false-memory classifier must agree with hand labels on **>= 90%** before any formation number is quoted (pre-registered gate).
2. If formation under M1 is ~0 even for silent/stale faults, the pre-registered branch applies: pivot the fault mix toward silent/stale before any scale-up.
3. File results per the project's Drive record + tracker routine.